In [1]:
# !wandb login --relogin
# 02fb2a333f28bd7bc2d3f3eee9661299bdb297c0


In [35]:
import tensorflow as tf
import wandb
from wandb.integration.keras import WandbCallback

# Configuración de hiperparámetros
config = dict(
    epochs=5,
    batch_size=128,
    learning_rate=0.005,
    architecture="CNN",
    dataset="MNIST"
)

In [36]:
# Definir el modelo CNN en Keras
def make_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax')
    ])
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=config['learning_rate']),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model


In [39]:
# Definir el modelo
def model_pipeline(hyperparameters):
    # Iniciar sesión en W&B
    with wandb.init(project="mnist-classification", config=hyperparameters):
        config = wandb.config
        
        # Preparar los datos
        (train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()
        train_images = train_images.reshape((-1, 28, 28, 1)).astype('float32') / 255
        test_images = test_images.reshape((-1, 28, 28, 1)).astype('float32') / 255
        
        # Crear el modelo
        model = make_model()
        
        # Entrenar el modelo
        model.fit(train_images, train_labels,
                  epochs=config.epochs,
                  batch_size=config.batch_size,
                  validation_data=(test_images, test_labels),
                  callbacks=[wandb.keras.WandbCallback()])

        # Evaluar el modelo
        test_loss, test_acc = model.evaluate(test_images, test_labels)
        print(f"Test accuracy: {test_acc * 100:.2f}%")

    return model



In [40]:
# Ejecutar el pipeline
if __name__ == "__main__":
    model_pipeline(config)

/home/canveo/Projects/notebook/wandb_train/wandb_env/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
Traceback (most recent call last):
  File "/tmp/ipykernel_4665/1035408488.py", line 16, in model_pipeline
    model.fit(train_images, train_labels,
  File "/home/canveo/Projects/notebook/wandb_train/wandb_env/lib/python3.10/site-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler
  File "/home/canveo/Projects/notebook/wandb_train/wandb_env/lib/python3.10/site-packages/wandb/integration/keras/keras.py", line 555, in set_model
AttributeError: can't set attribute 'model'


AttributeError: can't set attribute 'model'

In [41]:
import wandb
import random

# start a new wandb run to track this script
wandb.init(
    # set the wandb project where this run will be logged
    project="my-awesome-project",

    # track hyperparameters and run metadata
    config={
    "learning_rate": 0.02,
    "architecture": "CNN",
    "dataset": "CIFAR-100",
    "epochs": 10,
    }
)

# simulate training
epochs = 10
offset = random.random() / 5
for epoch in range(2, epochs):
    acc = 1 - 2 ** -epoch - random.random() / epoch - offset
    loss = 2 ** -epoch + random.random() / epoch + offset

    # log metrics to wandb
    wandb.log({"acc": acc, "loss": loss})

# [optional] finish the wandb run, necessary in notebooks
wandb.finish()

acc,▂▁▄▄▆█▆▇
loss,█▆▂▅▂▃▂▁
acc,0.79972
loss,0.18417


In [47]:
# This script needs these libraries to be installed:
#   tensorflow, numpy

import wandb
from wandb.keras import WandbMetricsLogger, WandbModelCheckpoint
from keras.callbacks import ModelCheckpoint

import random
import numpy as np
import tensorflow as tf


# Start a run, tracking hyperparameters
wandb.init(
    # set the wandb project where this run will be logged
    project="my-awesome-project",

    # track hyperparameters and run metadata with wandb.config
    config={
        "layer_1": 512,
        "activation_1": "relu",
        "dropout": random.uniform(0.01, 0.80),
        "layer_2": 10,
        "activation_2": "softmax",
        "optimizer": "sgd",
        "loss": "sparse_categorical_crossentropy",
        "metric": "accuracy",
        "epoch": 8,
        "batch_size": 256
    }
)

# [optional] use wandb.config as your config
config = wandb.config

# get the data
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
x_train, y_train = x_train[::5], y_train[::5]
x_test, y_test = x_test[::20], y_test[::20]
labels = [str(digit) for digit in range(np.max(y_train) + 1)]

# build a model
model = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(config.layer_1, activation=config.activation_1),
    tf.keras.layers.Dropout(config.dropout),
    tf.keras.layers.Dense(config.layer_2, activation=config.activation_2)
    ])

# compile the model
model.compile(optimizer=config.optimizer,
              loss=config.loss,
              metrics=[config.metric]
              )

# WandbMetricsLogger will log train and validation metrics to wandb
# WandbModelCheckpoint will upload model checkpoints to wandb
history = model.fit(x=x_train, y=y_train,
                    epochs=config.epoch,
                    batch_size=config.batch_size,
                    validation_data=(x_test, y_test),
                    callbacks=[
                      WandbMetricsLogger(log_freq=5),
                      # WandbModelCheckpoint("models")
                      # ModelCheckpoint(filepath="models/best_model.h5", save_best_only=True, monitor='val_loss')

                    ])

# [optional] finish the wandb run, necessary in notebooks
wandb.finish()

batch/accuracy,▁▃▃▄▅▅▆▇▇█
batch/batch_step,▁▂▃▃▄▅▆▆▇█
batch/loss,█▆▅▅▄▃▃▂▂▁
epoch/accuracy,▁
epoch/epoch,▁
epoch/loss,▁
epoch/val_accuracy,▁
epoch/val_loss,▁
batch/accuracy,0.20805
batch/batch_step,45
batch/loss,2.2138


Epoch 1/8


/home/canveo/Projects/notebook/wandb_train/wandb_env/lib/python3.10/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.


47/47 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.1491 - loss: 2.3435 - val_accuracy: 0.5980 - val_loss: 1.8332
Epoch 2/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.3740 - loss: 1.8918 - val_accuracy: 0.7220 - val_loss: 1.5214
Epoch 3/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5220 - loss: 1.5937 - val_accuracy: 0.7660 - val_loss: 1.2929
Epoch 4/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5909 - loss: 1.4036 - val_accuracy: 0.7860 - val_loss: 1.1263
Epoch 5/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6426 - loss: 1.2563 - val_accuracy: 0.8020 - val_loss: 1.0030
Epoch 6/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6766 - loss: 1.1332 - val_accuracy: 0.8140 - val_loss: 0.9087
Epoch 7/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 970us/step - accuracy: 0.6979 - loss: 1.0490 - val_accuracy: 0.8160 - val_loss: 0.8344
Epoch 8/8
47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7190 - loss: 0.9715 - val_accuracy: 0.8260 - val_loss: 0.7764


batch/accuracy,▁▁▂▂▂▄▄▄▄▄▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇█▇█████████████
batch/batch_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
batch/loss,██▇▇▇▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,▁▄▆▆▇▇██
epoch/epoch,▁▂▃▄▅▆▇█
epoch/loss,█▆▄▃▃▂▁▁
epoch/val_accuracy,▁▅▆▇▇███
epoch/val_loss,█▆▄▃▃▂▁▁
batch/accuracy,0.72155
batch/batch_step,395
batch/loss,0.96312
